In [ ]:
%load_ext autoreload
%autoreload 2


import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from pendulibrary.integrate import integrate_state
from pendulibrary.continuation import adaptive_cont
from pendulibrary.common_targetters import single_fixed
from pendulibrary.utils import get_x0_linear, get_x0_corrected

int_tol = 1e-14

In [ ]:
Lr = 1.0
Mr = 1.0

x0_ud, T_ud, tan_ud = get_x0_corrected(np.pi, 0.0, Lr, Mr, 1e-4)
x0_du, T_du, tan_du = get_x0_corrected(0.0, np.pi, Lr, Mr, 1e-4)
x0s_dd, Ts_dd, tans_dd = get_x0_corrected(0.0, 0.0, Lr, Mr, 1e-4)

In [ ]:
cont_kwargs = dict(
    s0=1e-4, s_min=1e-4, S=10.0, tol=1e-12, max_iter=6, target_iter=3, rate=1.05
)
targ = single_fixed(
    ind_fixed=0, val_fixed=np.pi, ind_no_enforce=2, Lr=Lr, Mr=Mr, int_tol=int_tol
)

func = targ.f_df_stm
X0, tan = targ.get_X(x0_ud, T_ud), tan_ud
Xs, eig_vals, (DFs, tangents, stms) = adaptive_cont(
    X0, func, tan, **cont_kwargs, exact_tangent=True
)

In [ ]:
s_vals = np.append(0, np.cumsum(np.linalg.norm(np.diff(Xs, axis=0), axis=1)))

In [ ]:
ts, xs, fs = integrate_state(
    targ.get_x0(Xs[-1]), targ.get_period(Xs[-1]), Lr, Mr, 1e-14
)

In [ ]:
y = -(np.cos(xs[0]) + Lr * np.cos(xs[1]))
x = np.sin(xs[0]) + Lr * np.sin(xs[1])
plt.plot(x, y)
plt.axis("equal")